# Mouse bacterial family comparisons

Figure 2 source analysis. Includes the supplied Welch comparisons, GraphPad table preparation and the source Python summary plot. The exact assembled panel and Prism project were unavailable.

In [ ]:
from pathlib import Path

# Jupyter normally starts the kernel in the notebook directory.
here = Path.cwd()
repo_root = next(parent for parent in (here, *here.parents)
                 if (parent / "requirements.txt").is_file()
                 and (parent / "mouse").is_dir() and (parent / "human").is_dir())
data_dir = repo_root / "data"
output_dir = repo_root / "outputs" / 'mouse/03_family_abundance'
output_dir.mkdir(parents=True, exist_ok=True)
notebook_output = output_dir


## Load the workbook and rank features

In [ ]:
import pandas as pd

# Define the file path
path = data_dir / 'Mouse_Data_For_PCA.xlsx'

# Read the Excel file into a DataFrame
df = pd.read_excel(path)
# Remove rows where all values are NaN
df = df.dropna(how='all')

df


In [ ]:
# Define label columns (metadata)
label_cols = ['Full ID', 'Group', 'Timepoint', 'Sex']

# Define feature columns (everything else)
feature_cols = [col for col in df.columns if col not in label_cols]

# Create separate DataFrames
df_labels = df[label_cols]
df_features = df[feature_cols]


In [ ]:
# ============================================================
# PCA Visualization with Linear Boundary & Accuracy Annotation
# Save Top 10 Differential Features
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# -------------------------------
# Global style (font + scaling)
# -------------------------------
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,            # Base font size
    'axes.titlesize': 18,       # Title size
    'axes.labelsize': 16,       # Axis label size
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'legend.title_fontsize': 14,
    'figure.titlesize': 20
})

plt.style.use("default")

plt.rcParams['pdf.fonttype'] = 42       # Use TrueType fonts (not Type 3)
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'   # Keep text editable in SVGs

# -------------------------------
# Ensure group order
# -------------------------------
df_labels["Group"] = pd.Categorical(
    df_labels["Group"],
    categories=["B6", "M83(ho)"],
    ordered=True
)

# -------------------------------
# Compute global Z-scores
# -------------------------------
scaler = StandardScaler()
df_z = pd.DataFrame(
    scaler.fit_transform(df_features),
    columns=df_features.columns,
    index=df_features.index
)

# -------------------------------
# Compute mean Z-score per group
# -------------------------------
df_mean_z = df_z.groupby(df_labels["Group"]).mean().T
df_mean_z["Difference"] = df_mean_z["B6"] - df_mean_z["M83(ho)"]

# -------------------------------
# Filter features by absolute difference threshold
# -------------------------------
threshold = 0.4
selected_features = df_mean_z.loc[df_mean_z["Difference"].abs() > threshold].index

num_selected = len(selected_features)
num_total = df_features.shape[1]
print(f"Kept {num_selected} of {num_total} taxa ({num_selected / num_total:.1%}) with |ΔZ| > {threshold}")

# -------------------------------
# Identify Top 10 absolute differences
# -------------------------------
top10_features = (
    df_mean_z.reindex(selected_features)
    .loc[:, "Difference"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top10_features = list(top10_features)
print("\nTop 10 most discriminative features (by |ΔZ|):")
for i, f in enumerate(top10_features, 1):
    print(f"{i}. {f}")

# -------------------------------
# PCA on filtered features
# -------------------------------


## Screen candidate families with Welch’s t-test

In [ ]:
# ============================================================
# Top 10 Features (Taxa) – Bar + Scatter + Significance Stars
# + Store Significant Differences
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import ttest_ind

# --- Prepare data ---
df_plot = df_features[top10_features].copy()
df_plot["Group"] = df_labels["Group"].values

# --- Styling ---

palette = {"B6": "#42A5F5", "M83(ho)": "#EF5350"}

# Helper to turn p-values into stars
def p_to_star(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# Dictionary to store significant taxa results
sig_dict = {}

for feature in top10_features:
    # --- Compute t-test between groups ---
    b6 = df_plot.loc[df_plot["Group"] == "B6", feature]
    m83 = df_plot.loc[df_plot["Group"] == "M83(ho)", feature]
    tstat, pval = ttest_ind(b6, m83, equal_var=False, nan_policy='omit')

    # Compute mean difference (M83 - B6)
    diff = m83.mean() - b6.mean()

    star = p_to_star(pval)

    # --- Store significance results ---
    if pval < 0.05:
        direction = "increased in M83(ho)" if diff > 0 else "decreased in M83(ho)"
        sig_dict[feature] = {
            "p_value": pval,
            "significance": star,
            "direction": direction
        }


## Prepare GraphPad tables and inspect the source stratified statistics

In [ ]:
# ============================================================
# GraphPad CSV Export for Top 10 Families
# Each file handles variable sample sizes per block
# ============================================================

import pandas as pd
import numpy as np
import os

# --- Setup ---
output_dir = notebook_output / 'Fig2_CSVs'
os.makedirs(output_dir, exist_ok=True)

# --- Define export function ---
def make_graphpad_ready(df_sub, feature, n_gap=2, max_animals=12):
    """
    Create GraphPad-style CSV:
    B6(males) | 2*NaN | B6(females) | 2*NaN | M83(males) | 2*NaN | M83(females)
    Each block padded up to 'max_animals' columns.
    """
    output_rows = []

    def get_vals(gr, sex, tp):
        vals = df_sub.loc[
            (df_sub["Group"] == gr)
            & (df_sub["Sex"].str.lower().str.contains(sex))
            & (df_sub["Timepoint"] == tp),
            feature,
        ].dropna().tolist()
        return vals

    for tp in ["2 Months", "4 Months"]:
        row = [tp]

        b6_male = get_vals("B6", "male", tp)
        b6_fem = get_vals("B6", "female", tp)
        m83_male = get_vals("M83(ho)", "male", tp)
        m83_fem = get_vals("M83(ho)", "female", tp)

        def pad(vals):
            return vals + [np.nan] * (max_animals - len(vals))

        row.extend(
            pad(b6_male)
            + [np.nan] * n_gap
            + pad(b6_fem)
            + [np.nan] * n_gap
            + pad(m83_male)
            + [np.nan] * n_gap
            + pad(m83_fem)
        )
        output_rows.append(row)

    # build header list
    headers = ["Timepoint"]
    group_labels = [
        ("B6_male", max_animals),
        ("gap1", n_gap),
        ("B6_fem", max_animals),
        ("gap2", n_gap),
        ("M83_male", max_animals),
        ("gap3", n_gap),
        ("M83_fem", max_animals),
    ]
    for name, count in group_labels:
        headers.extend([f"{name}_{i+1}" for i in range(count)])

    expected_len = len(output_rows[0])
    if len(headers) != expected_len:
        headers += [f"extra_{i+1}" for i in range(expected_len - len(headers))]

    return pd.DataFrame(output_rows, columns=headers)


# --- Loop over each family feature and export ---


In [ ]:
# --- Initialize dictionary to hold all GraphPad-ready DataFrames ---
graphpad_dict = {}

for feature in top10_features:
    # Combine feature data with metadata
    df_base = df_features[[feature]].copy()
    df_base["Group"] = df_labels["Group"].values
    df_base["Sex"] = df_labels["Sex"].values
    df_base["Timepoint"] = df_labels["Timepoint"].values

    # Convert to GraphPad format
    df_out = make_graphpad_ready(df_base, feature, n_gap=2, max_animals=12)

    # Store in dictionary
    graphpad_dict[feature] = df_out

    # Optionally export to CSV
    outfile = os.path.join(output_dir, f"{feature}_GraphPad.csv")
    df_out.to_csv(outfile, index=False)
    # print(f"✅  Saved CSV for: {feature}")

# Optional confirmation
print(f"📦 Stored {len(graphpad_dict)} DataFrames in 'graphpad_dict'")


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

results = []

def extract_values(df, group_prefix):
    vals = df.filter(like=group_prefix).values.flatten()
    return vals[~np.isnan(vals)]

for family, df in graphpad_dict.items():
    for tp in ["2 Months", "4 Months"]:
        df_tp = df[df["Timepoint"] == tp].copy()

        # --- Define subsets ---
        subsets = {
            "All": {
                "B6": df_tp.filter(like="B6_"),
                "M83": df_tp.filter(like="M83_")
            },
            "Males": {
                "B6": df_tp.filter(like="B6_male"),
                "M83": df_tp.filter(like="M83_male")
            },
            "Females": {
                "B6": df_tp.filter(like="B6_fem"),
                "M83": df_tp.filter(like="M83_fem")
            },
        }

        for subset_name, groups in subsets.items():
            b6_vals = extract_values(groups["B6"], "B6")
            m83_vals = extract_values(groups["M83"], "M83")

            # --- Skip if not enough data ---
            if len(b6_vals) < 3 or len(m83_vals) < 3:
                continue

            # --- Normality ---
            b6_p = stats.shapiro(b6_vals).pvalue
            m83_p = stats.shapiro(m83_vals).pvalue
            both_normal = (b6_p > 0.05) and (m83_p > 0.05)

            # --- Test choice ---
            if both_normal:
                test_name = "Student t-test"
                stat, p_val = stats.ttest_ind(b6_vals, m83_vals, equal_var=True)
            else:
                test_name = "Welch’s t-test"
                stat, p_val = stats.ttest_ind(b6_vals, m83_vals, equal_var=False)

            results.append({
                "Family": family,
                "Timepoint": tp,
                "Subset": subset_name,
                "B6_N": len(b6_vals),
                "M83_N": len(m83_vals),
                "B6_Shapiro_p": b6_p,
                "M83_Shapiro_p": m83_p,
                "Test": test_name,
                "Statistic": stat,
                "p-value": p_val
            })

# --- Build results DataFrame ---
stats_summary = pd.DataFrame(results).sort_values(["Family", "Timepoint", "Subset"])

print("\n📊 Statistical Summary by Timepoint and Sex\n")
print(stats_summary.to_string(index=False))


## Prepare the 2- and 4-month GraphPad tables

In [ ]:
# ============================================================
# Correlation of Significant Taxa with Timepoint (M83(ho) only)
# + Heatmap Visualization (all correlations shown, outlined numbers)
# ============================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import patheffects
from scipy.stats import pearsonr

# --- Keep only M83(ho) participants ---
df_m83 = df_features.copy()
df_m83["Group"] = df_labels["Group"].values
df_m83 = df_m83[df_m83["Group"] == "M83(ho)"].copy()

if "Sex" in df_labels.columns:
    df_m83["Sex"] = df_labels.loc[df_m83.index, "Sex"].values

# --- Clinical variable(s) ---
clinical_vars = ["Timepoint"]

# Attach this column from df_labels
for var in clinical_vars:
    df_m83[var] = df_labels.loc[df_m83.index, var].values

# 🔧 Extract numeric month value from strings like "2 months", "4 months"
df_m83["Timepoint"] = (
    df_m83["Timepoint"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)   # extract digits only
    .astype(float)                         # convert to numeric
)

df_m83_full = df_m83.copy()


In [ ]:
# ============================================================
# Top 4 Significant Taxa – Bar + Scatter + Significance (M83(ho): 2 vs 4 months)
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import ttest_ind

# --- Parameters ---
top_n = 4  # number of features to show

# --- Prepare data ---
top_features = list(sig_dict.keys())[:top_n]


In [ ]:
import os
import numpy as np
import pandas as pd

# Output folder for CSVs
output_dir = notebook_output / 'Fig2_GraphPad_Outputs'
os.makedirs(output_dir, exist_ok=True)

for feature in top_features:
    # Extract relevant columns
    df_feat = df_m83_full[["Timepoint", feature, "Sex"]].dropna(subset=[feature])

    # Rename for clarity
    df_feat = df_feat.rename(columns={"Timepoint": "Month", feature: "Value"})

    # Split by sex
    df_males = df_feat[df_feat["Sex"].str.lower().str.startswith("m")].copy()
    df_females = df_feat[df_feat["Sex"].str.lower().str.startswith("f")].copy()

    # Group and pivot for each sex section
    def make_section(df):
        """Return section as DataFrame with columns 2m,4m"""
        t2 = df.loc[df["Month"] == 2, "Value"].reset_index(drop=True)
        t4 = df.loc[df["Month"] == 4, "Value"].reset_index(drop=True)
        n = max(len(t2), len(t4))
        out = pd.DataFrame({
            "2_months": np.pad(t2, (0, n - len(t2)), constant_values=np.nan),
            "4_months": np.pad(t4, (0, n - len(t4)), constant_values=np.nan),
        })
        return out

    males_section = make_section(df_males)
    females_section = make_section(df_females)

    # Add one blank row between males/females
    gap = pd.DataFrame({"2_months": [np.nan], "4_months": [np.nan]})

    df_out = pd.concat([males_section, gap, females_section], ignore_index=True)

    # Rename columns for GraphPad (customize if needed)
    df_out.columns = ["2 months", "4 months"]

    # Save CSV for this feature
    safe_name = feature.replace("/", "_").replace("\\", "_").replace(":", "_")
    save_path = os.path.join(output_dir, f"{safe_name}_2v4_Months.csv")


    df_out.to_csv(save_path, index=False, float_format="%.5g")
    print(f"Saved → {save_path}")

df_out


## Plot genotype and timepoint summaries

In [ ]:
# ============================================================
# Summary Figure: Top 4 Differential Taxa + M83(ho) 2‑vs‑4 Month Comparison
# (Only first column shows “Abundance” on y‑axis)
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

# ============================================================
# ▪ Global Style (PCA‑consistent)
# ============================================================
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'legend.title_fontsize': 14,
    'figure.titlesize': 20,
    'axes.edgecolor': 'black',
    'axes.linewidth': 1.2,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white'
})

plt.style.use("default")
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

palette_groups = {"B6": "#42A5F5", "M83(ho)": "#EF5350"}
palette_time = {2.0: "#42A5F5", 4.0: "#EF5350"}

# ============================================================
# ▪ Identify Top 4 differential taxa
# ============================================================
sig_df = (
    pd.DataFrame.from_dict(sig_dict, orient="index")
      .reset_index()
      .rename(columns={"index": "Taxon"})
      .sort_values("p_value")
)
top4_taxa = sig_df["Taxon"].head(4).tolist()

# --- Prepare data ---
df_combo = df_features[top4_taxa].copy()
df_combo["Group"] = df_labels["Group"].values
df_combo["Sex"] = df_labels["Sex"].str.lower()
df_combo["Timepoint"] = (
    df_labels["Timepoint"]
    .astype(str)
    .str.extract(r"(\d+)").astype(float)
)
df_m83 = df_combo[df_combo["Group"] == "M83(ho)"].copy()

# ============================================================
# ▪ Layout setup
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.reshape(2, 4)

bar_kws = dict(
    dodge=False,
    errorbar="se",
    alpha=0.9,
    width=0.6,
    linewidth=2.2,
    edgecolor="black"
)

# ============================================================
# ▪ Top row: B6 vs M83(ho)
# ============================================================
for i, taxon in enumerate(top4_taxa):
    ax = axes[0, i]

    sns.barplot(
        data=df_combo, x="Group", y=taxon,
        hue="Group", palette=palette_groups,
        legend=False, ax=ax, **bar_kws
    )

    for grp in palette_groups.keys():
        sub = df_combo[df_combo["Group"] == grp]

        # Males → black filled
        male_sub = sub[sub["Sex"].isin(["male", "m"])]
        sns.stripplot(
            data=male_sub, x="Group", y=taxon,
            color="black", edgecolor="black",
            size=6, jitter=0.15, alpha=0.9, ax=ax
        )

        # Females → transparent w/ black edge
        female_sub = sub[sub["Sex"].isin(["female", "f"])]
        sns.stripplot(
            data=female_sub, x="Group", y=taxon,
            facecolor="white", edgecolor="black",
            linewidth=1.3, marker="o",
            size=6, jitter=0.15, alpha=1, ax=ax
        )

    # --- Significance stars
    star = sig_dict[taxon]["significance"]
    group1 = df_combo.loc[df_combo["Group"] == "B6", taxon]
    group2 = df_combo.loc[df_combo["Group"] == "M83(ho)", taxon]
    y_max, y_min = max(group1.max(), group2.max()), min(group1.min(), group2.min())
    y_range = y_max - y_min
    y_pos = y_max + 0.12 * y_range
    ax.set_ylim(y_min - 0.05 * y_range, y_max + 0.25 * y_range)
    ax.plot([0, 0, 1, 1], [y_pos*0.985, y_pos, y_pos, y_pos*0.985], lw=1.8, c='black')
    ax.text(0.5, y_pos, star,
            ha='center', va='bottom', color='black',
            fontsize=16, fontweight='bold')

    ax.set_title(f"{taxon}", pad=6)
    ax.set_xlabel("Group", fontsize=16)

    # ✅ Only show y-label on first (leftmost) column
    if i == 0:
        ax.set_ylabel("Relative Abundance", fontsize=16)
    else:
        ax.set_ylabel("")

    ax.tick_params(labelsize=13)

# ============================================================
# ▪ Bottom row: M83(ho) 2 vs 4 months
# ============================================================
def p_to_star(p):
    if p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

for i, taxon in enumerate(top4_taxa):
    ax = axes[1, i]
    sns.barplot(
        data=df_m83, x="Timepoint", y=taxon,
        hue="Timepoint", palette=palette_time,
        legend=False, ax=ax, **bar_kws
    )

    for tp in palette_time.keys():
        sub = df_m83[df_m83["Timepoint"] == tp]
        male_sub = sub[sub["Sex"].isin(["male", "m"])]
        sns.stripplot(
            data=male_sub, x="Timepoint", y=taxon,
            color="black", edgecolor="black",
            size=6, jitter=0.15, alpha=0.9, ax=ax
        )
        female_sub = sub[sub["Sex"].isin(["female", "f"])]
        sns.stripplot(
            data=female_sub, x="Timepoint", y=taxon,
            facecolor="white", edgecolor="black",
            linewidth=1.3, marker="o", size=6,
            jitter=0.15, alpha=1, ax=ax
        )

    # t‑test stats
    t2 = df_m83.loc[df_m83["Timepoint"] == 2.0, taxon]
    t4 = df_m83.loc[df_m83["Timepoint"] == 4.0, taxon]
    if len(t2) >= 2 and len(t4) >= 2:
        tstat, pval = ttest_ind(t2, t4, equal_var=False, nan_policy='omit')
        star = p_to_star(pval)
    else:
        star, pval = "n/a", np.nan

    y_max, y_min = df_m83[taxon].max(), df_m83[taxon].min()
    y_rng = y_max - y_min
    y_pos = y_max + 0.12 * y_rng
    ax.set_ylim(y_min - 0.05 * y_rng, y_max + 0.25 * y_rng)
    ax.plot([0, 0, 1, 1], [y_pos*0.985, y_pos, y_pos, y_pos*0.985], lw=1.8, c='black')
    ax.text(0.5, y_pos, star,
            ha='center', va='bottom', color='black',
            fontsize=16, fontweight='bold')

    ax.set_title(f"{taxon}", pad=6)
    ax.set_xlabel("Timepoint (M83 mice only)", fontsize=16)

    # ✅ Only show y-label in first column
    if i == 0:
        ax.set_ylabel("Relative Abundance", fontsize=16)
    else:
        ax.set_ylabel("")

    ax.set_xticklabels(["2 months", "4 months"])
    ax.tick_params(labelsize=13)

# ============================================================
# ▪ Final layout
# ============================================================
for j in range(len(top4_taxa), 4):
    fig.delaxes(axes[0, j])
    fig.delaxes(axes[1, j])

fig.suptitle(
    "Summary: Top 4 Differential Taxa — B6 vs M83(ho) and 2 vs 4 Month Changes",
    fontsize=20, y=1.05, fontweight='bold'
)

plt.tight_layout()
plt.subplots_adjust(hspace=0.4)

output_path = notebook_output
# plt.savefig(output_path / "Mouse_Correlation_Fig_2.pdf", format="pdf", bbox_inches="tight")

plt.show()
